# Tải bộ dữ liệu ung thư phổi GSE68465 từ GEO

Notebook này tải trực tiếp file `series_matrix.txt.gz` từ FTP của NCBI, tách ra
ma trận biểu hiện và bảng metadata, tự động gán nhãn giai đoạn (cột `stage`,
4 giá trị: `Normal`/`I`/`II`/`III` — cohort này không có mẫu stage IV), số hoá
nhãn thành cột số (`stage_encoded`, `stage_binary_encoded`) rồi lưu **cả nhãn
lẫn ma trận biểu hiện vào 1 file H5 duy nhất** (mỗi GSE có 2 key: `<GSE>/expr`
và `<GSE>/pheno`) lên Google Drive — không cần đọc thêm file CSV riêng nữa.
Một bản CSV (`geo_labels.csv`) vẫn được ghi thêm để xem nhanh, nhưng không còn
là nguồn dữ liệu bắt buộc. Thiết kế để chạy không cần can thiệp tay trên Colab.

Bộ dữ liệu: GSE68465


## 1. Thiết lập

In [ ]:
import os, re, io, gzip, urllib.request, warnings
import pandas as pd
import numpy as np

warnings.filterwarnings("ignore")

GSE_LIST = ["GSE68465"]

OUTDIR = "/content/geo_data"
RAWDIR = os.path.join(OUTDIR, "raw")
os.makedirs(RAWDIR, exist_ok=True)

print("Thu muc luu (tam, local):", OUTDIR)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_OUTDIR = "/content/drive/MyDrive/KLTN/data/geo"
os.makedirs(DRIVE_OUTDIR, exist_ok=True)
print("Thu muc luu tren Drive:", DRIVE_OUTDIR)

!pip install -q tables

## 2. Hàm tải và phân tích series matrix

`series_matrix.txt.gz` gồm hai phần: các dòng bắt đầu bằng `!Sample_` là metadata,
và phần bảng nằm giữa `!series_matrix_table_begin` / `_end` là ma trận biểu hiện.

Một series có thể bị chia thành nhiều file nếu dùng nhiều platform, nên hàm dưới
liệt kê thư mục FTP trước rồi tải hết.

In [ ]:
def matrix_file_urls(gse):
    """Liet ke tat ca file series_matrix cua mot GSE tren FTP NCBI."""
    stub = gse[:-3] + "nnn"
    base = f"https://ftp.ncbi.nlm.nih.gov/geo/series/{stub}/{gse}/matrix/"
    with urllib.request.urlopen(base, timeout=120) as r:
        html = r.read().decode("utf-8", errors="replace")
    files = sorted(set(re.findall(r'href="([^"]*series_matrix\.txt\.gz)"', html)))
    if not files:
        raise RuntimeError(f"Khong tim thay series matrix cho {gse}")
    return [base + f for f in files]


def download(url, dest_dir):
    fname = url.rsplit("/", 1)[-1]
    path = os.path.join(dest_dir, fname)
    if os.path.exists(path) and os.path.getsize(path) > 0:
        return path
    urllib.request.urlretrieve(url, path)
    return path


def parse_series_matrix(path):
    """Tra ve (expr, pheno). expr: probe x sample. pheno: sample x metadata."""
    meta, table_lines, in_table = {}, [], False

    with gzip.open(path, "rt", encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.rstrip("\n").rstrip("\r")
            if line.startswith("!series_matrix_table_begin"):
                in_table = True
                continue
            if line.startswith("!series_matrix_table_end"):
                in_table = False
                continue
            if in_table:
                if line:
                    table_lines.append(line)
            elif line.startswith("!Sample_"):
                parts = line.split("\t")
                key = parts[0][1:]
                vals = [v.strip().strip('"') for v in parts[1:]]
                meta.setdefault(key, []).append(vals)

    # Mot so key (vd Sample_characteristics_ch1) xuat hien nhieu lan -> danh so
    flat = {}
    for key, occurrences in meta.items():
        if len(occurrences) == 1:
            flat[key] = occurrences[0]
        else:
            for i, occ in enumerate(occurrences, 1):
                flat[f"{key}_{i}"] = occ

    n = max(len(v) for v in flat.values())
    flat = {k: (v + [None] * (n - len(v))) for k, v in flat.items()}
    pheno = pd.DataFrame(flat)
    if "Sample_geo_accession" in pheno.columns:
        pheno = pheno.set_index("Sample_geo_accession")
    pheno.index.name = "gsm"

    expr = pd.read_csv(io.StringIO("\n".join(table_lines)),
                       sep="\t", index_col=0, low_memory=False)
    expr.index.name = "probe"
    expr = expr.apply(pd.to_numeric, errors="coerce")
    expr = expr.loc[:, [c for c in expr.columns if c in pheno.index]]
    return expr, pheno

## 3. Tải bộ dữ liệu

In [ ]:
datasets = {}

for gse in GSE_LIST:
    print(f"\n=== {gse} ===")
    try:
        urls = matrix_file_urls(gse)
    except Exception as e:
        print("  LOI khi liet ke file:", e)
        continue

    exprs, phenos = [], []
    for url in urls:
        print("  tai", url.rsplit("/", 1)[-1], "...", end=" ")
        try:
            path = download(url, RAWDIR)
            e, p = parse_series_matrix(path)
            exprs.append(e)
            phenos.append(p)
            print(f"{e.shape[0]} probe x {e.shape[1]} mau")
        except Exception as err:
            print("LOI:", err)

    if not exprs:
        continue

    expr = pd.concat(exprs, axis=1) if len(exprs) > 1 else exprs[0]
    pheno = pd.concat(phenos, axis=0) if len(phenos) > 1 else phenos[0]
    datasets[gse] = {"expr": expr, "pheno": pheno}
    print(f"  -> TONG: {expr.shape[0]} probe x {expr.shape[1]} mau")

## 4. Tìm cột chứa thông tin giai đoạn

Mỗi bộ ghi nhãn stage ở một trường khác nhau và bằng cách viết khác nhau.
Cell này quét toàn bộ metadata để tìm những cột có nhắc tới "stage", rồi in
vài giá trị mẫu. **Bạn phải tự đọc kết quả này** trước khi tin vào bước chuẩn hoá
ở cell sau.

In [ ]:
def find_stage_columns(pheno):
    hits = []
    for c in pheno.columns:
        vals = pheno[c].astype(str)
        if vals.str.contains("stage", case=False, na=False).any():
            hits.append(c)
    return hits


for gse, d in datasets.items():
    pheno = d["pheno"]
    cols = find_stage_columns(pheno)
    print(f"\n=== {gse} ===")
    if not cols:
        print("  KHONG tim thay cot nao chua 'stage' -- kiem tra thu cong:")
        cc = [c for c in pheno.columns if "characteristics" in c.lower()]
        for c in cc[:8]:
            print(f"    {c}: {pheno[c].dropna().unique()[:3]}")
        continue
    for c in cols:
        uniq = pheno[c].dropna().unique()
        print(f"  [{c}] {len(uniq)} gia tri khac nhau")
        for u in uniq[:8]:
            print("      ", u)

## 5. Chuẩn hoá nhãn giai đoạn

Hàm dưới xử lý các kiểu viết phổ biến (`stage: IA`, `Stage IIIB`, `tnm stage: 1`,
`pathological stage: II`). Nó **không phải giải pháp vạn năng** — sau khi chạy,
hãy đối chiếu bảng phân bố ở cuối với những gì bạn thấy ở cell 4. Bộ nào ra toàn
`None` thì cần viết mapping riêng.

In [ ]:
ARABIC_TO_ROMAN = {"1": "I", "2": "II", "3": "III", "4": "IV"}

MISSING = {"", "na", "n/a", "nan", "none", "null", "unknown",
           "not available", "not reported", "---", "-", "."}


def stage_from_tnm_string(s):
    """
    Mot so series (vd GSE68465) khong ghi 'Stage IIIA' ma chi ghi ma pTNM,
    vi du 'disease_stage: pN0pT1'. Ham nay doc truc tiep tu pT/pN va suy ra
    stage AJCC tho (I/II/III), THEO GIA DINH M0 (chi dung cho cohort da
    phau thuat cat bo, khong co truong metastasis rieng -- neu series co
    truong M/metastasis rieng thi KHONG dung ham nay, phai xu ly ca M).

    Quy tac (AJCC NSCLC, M0):
      N2 hoac N3, hoac T4, hoac (T3 va N1)  -> III
      T3 hoac N1 (con lai)                  -> II
      con lai (T1-2, N0)                    -> I

    Tra ve None neu khong tim thay ca pT lan pN trong chuoi.
    """
    t_match = re.search(r"p?t\s*([0-4])", s)
    n_match = re.search(r"p?n\s*([0-3])", s)
    if not t_match or not n_match:
        return None
    t, n = int(t_match.group(1)), int(n_match.group(1))
    if n >= 2 or t >= 4 or (t == 3 and n == 1):
        return "III"
    if t == 3 or n == 1:
        return "II"
    return "I"


def normalize_stage(x):
    """'pathological stage: IIIB' -> 'III'. Tra ve None neu khong doc duoc."""
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return None
    s = str(x).strip().lower()
    if s in MISSING:
        return None

    # Dinh dang ma pTNM rieng (vd 'disease_stage: pn0pt1') -- KHONG co chu
    # "stage <so/la ma>" chuan nen phai bat truoc, tranh de doan duoi day
    # (von danh cho chuoi "Stage III...") xoa mat pT/pN roi tra ve None oan.
    # Dieu kien ca pt VA pn cung xuat hien de tranh nham voi text khac.
    if re.search(r"p?t\s*[0-4]", s) and re.search(r"p?n\s*[0-3]", s):
        tnm_stage = stage_from_tnm_string(s)
        if tnm_stage is not None:
            return tnm_stage

    # bo moi thu truoc tu "stage", vd "pathological stage: iiia" -> "iiia"
    s = re.sub(r"^.*stage\s*[:=]?\s*", "", s).strip()
    if s in MISSING:
        return None
    # stage 0 (AIS / in situ) -- giu lai de khong mat am tham
    if re.match(r"0(?![0-9])", s):
        return "0"
    # So la ma. KHONG dung \b vi "ia" se khong khop.
    # Dieu kien: khong duoc theo sau boi mot ky tu la ma khac.
    m = re.match(r"(iv|iii|ii|i)(?![ivx])", s)
    if m:
        return m.group(1).upper()
    # So a rap, cho phep hau to chu cai: "1a", "2b"
    m = re.match(r"([1-4])(?![0-9])", s)
    if m:
        return ARABIC_TO_ROMAN[m.group(1)]
    return None


def binarize(stage):
    if stage in ("I", "II"):
        return "early"
    if stage in ("III", "IV"):
        return "advanced"
    return None


# Chon cot stage dau tien tim duoc cho moi bo. Sua tay o day neu chon sai.
STAGE_COL = {}
for gse, d in datasets.items():
    cols = find_stage_columns(d["pheno"])
    STAGE_COL[gse] = cols[0] if cols else None

print("Cot stage duoc chon (sua tay neu can):")
for k, v in STAGE_COL.items():
    print(f"  {k}: {v}")

In [ ]:
def find_disease_state_col(pheno):
    """
    Tim cot ghi ro mau la mo thuong hay khoi u (vd 'disease_state: Normal' /
    'disease_state: Lung Adenocarcinoma'), KHAC voi cot stage (vd 'disease_stage').

    Dung de phan biet hai truong hop deu ra stage=NaN nhung y nghia khac nhau:
      - mau mo thuong that (disease_state = Normal)              -> gan "Normal"
      - khoi u nhung khong doc duoc ma giai doan (vd 'pNXpT1')    -> KHONG phai Normal,
        loai khoi dataset vi khong the gan nhan dang tin cay.
    """
    hits = []
    for c in pheno.columns:
        if "stage" in c.lower():
            continue
        vals = pheno[c].astype(str).str.lower()
        if vals.str.contains(r"\bnormal\b", na=False).any():
            hits.append(c)
    return hits


# Chon cot disease_state dau tien tim duoc cho moi bo. Sua tay o day neu chon sai.
DISEASE_STATE_COL = {}
for gse, d in datasets.items():
    cols = find_disease_state_col(d["pheno"])
    DISEASE_STATE_COL[gse] = cols[0] if cols else None

print("Cot disease_state duoc chon (sua tay neu can):")
for k, v in DISEASE_STATE_COL.items():
    print(f"  {k}: {v}")

# Bang so hoa nhan, dung chung cho moi GSE. Co san ca ma cho "IV" du cohort
# GSE68465 khong co mau nao, de nhat quan neu sau nay them GSE khac co stage IV.
STAGE_ORDER = ["Normal", "I", "II", "III", "IV"]
STAGE_TO_CODE = {s: i for i, s in enumerate(STAGE_ORDER)}
BINARY_TO_CODE = {"early": 0, "advanced": 1}

In [ ]:
summary = []

for gse, d in datasets.items():
    pheno = d["pheno"].copy()
    col = STAGE_COL.get(gse)
    if col is None:
        pheno["stage"] = None
    else:
        pheno["stage"] = pheno[col].map(normalize_stage)

    # stage=NaN co 2 nghia khac nhau, khong duoc gop chung:
    #   - mau mo thuong that (disease_state = Normal)                 -> "Normal"
    #   - khoi u nhung khong doc duoc ma giai doan (vd 'pNXpT1', 'pp') -> loai khoi
    #     dataset, vi gan "Normal" cho mau ung thu la SAI nhan.
    state_col = DISEASE_STATE_COL.get(gse)
    if state_col is not None:
        is_true_normal = pheno[state_col].astype(str).str.contains("normal", case=False, na=False)
    else:
        is_true_normal = pd.Series(False, index=pheno.index)

    ambiguous = pheno["stage"].isna() & ~is_true_normal
    n_ambiguous = int(ambiguous.sum())
    if n_ambiguous:
        print(f"  [{gse}] loai {n_ambiguous} mau la khoi u nhung khong xac dinh duoc "
              f"giai doan (khong phai Normal): {list(pheno.index[ambiguous])}")
        pheno = pheno.loc[~ambiguous]
        d["expr"] = d["expr"].loc[:, pheno.index]

    pheno["stage"] = pheno["stage"].fillna("Normal")  # phan con lai: NaN that do la mo thuong
    pheno["stage_binary"] = pheno["stage"].map(binarize)  # None cho "Normal", chi co gia tri voi I-IV

    # So hoa nhan de dung thang cho ML, khong can LabelEncoder o notebook khac nua
    pheno["stage_encoded"] = pheno["stage"].map(STAGE_TO_CODE).astype(int)
    pheno["stage_binary_encoded"] = pheno["stage_binary"].map(BINARY_TO_CODE)  # NaN cho "Normal"

    d["pheno"] = pheno

    counts = pheno["stage"].value_counts(dropna=False).to_dict()
    summary.append({
        "GSE": gse,
        "n_sample": d["expr"].shape[1],
        "n_probe": d["expr"].shape[0],
        "n_excluded_ambiguous": n_ambiguous,
        "I": counts.get("I", 0), "II": counts.get("II", 0),
        "III": counts.get("III", 0), "IV": counts.get("IV", 0),
        "Normal": counts.get("Normal", 0),
    })

pd.DataFrame(summary).set_index("GSE")

## 6. Kiểm tra thang đo và platform

Một số series lưu giá trị đã log2, số khác lưu thang tuyến tính. Trộn hai loại
mà không xử lý sẽ phá hỏng mọi thứ. Cell này đoán bằng khoảng giá trị và in ra
platform của từng bộ.

In [ ]:
for gse, d in datasets.items():
    expr = d["expr"]
    vmax = np.nanmax(expr.values)
    vmin = np.nanmin(expr.values)
    logged = vmax < 100
    plat = d["pheno"].get("Sample_platform_id")
    plat = sorted(set(plat.dropna())) if plat is not None else ["?"]
    print(f"{gse:10s} platform={','.join(plat):12s} "
          f"min={vmin:9.2f} max={vmax:12.2f} "
          f"-> {'da log2' if logged else 'THANG TUYEN TINH, can log2'}")

In [ ]:
# Ap dung log2 cho nhung bo con o thang tuyen tinh
for gse, d in datasets.items():
    expr = d["expr"]
    if np.nanmax(expr.values) >= 100:
        d["expr"] = np.log2(expr.clip(lower=0) + 1)
        print(gse, "-> da chuyen sang log2")

## 7. Gộp nhãn + lưu vào 1 file H5 (kèm CSV để xem nhanh) lên Google Drive

Cột nhãn `stage` (+ bản số hoá `stage_encoded`, `stage_binary_encoded`) đã được
gán và lọc tự động ở bước 5, không cần chỉnh tay. Bảng nhãn được lưu **thẳng
vào file H5** dưới key `<GSE>/pheno`, cạnh ma trận biểu hiện `<GSE>/expr` —
notebook khác chỉ cần `pd.read_hdf(H5_PATH, key="<GSE>/pheno")` là có đủ nhãn,
không bắt buộc phải đọc CSV nữa (CSV vẫn được ghi thêm để xem nhanh).

In [ ]:
H5_PATH = os.path.join(DRIVE_OUTDIR, "geo_expr.h5")
CSV_PATH = os.path.join(DRIVE_OUTDIR, "geo_labels.csv")

all_pheno = []
with pd.HDFStore(H5_PATH, mode="w") as store:
    for gse, d in datasets.items():
        store.put(f"{gse}/expr", d["expr"])
        pheno = d["pheno"].copy()
        pheno["dataset"] = gse
        store.put(f"{gse}/pheno", pheno)  # nhan (stage, stage_encoded, ...) luu thang trong H5
        all_pheno.append(pheno)

labels = pd.concat(all_pheno, axis=0)
labels.to_csv(CSV_PATH)  # ban CSV chi de xem nhanh, khong bat buoc phai dung nua

print("Da luu:")
print(" ", H5_PATH, "(moi GSE co 2 key: '<GSE>/expr' va '<GSE>/pheno')")
print(" ", CSV_PATH, labels.shape)
print(labels["stage"].value_counts(dropna=False))
print(labels["stage_encoded"].value_counts(dropna=False).sort_index())
print("Ma so hoa stage:", STAGE_TO_CODE)

## 8. (Tuỳ chọn) Ánh xạ probe sang gene symbol

Ma trận ở trên là **probe ID**, không phải tên gene. Muốn gộp nhiều platform thì
bắt buộc phải quy về gene symbol trước.

Cell dưới dùng `GEOparse` để tải bảng chú giải của platform. Tên cột chứa gene
symbol khác nhau tuỳ platform (`Gene Symbol` với Affymetrix, `ILMN_Gene` hoặc
`Symbol` với Illumina) nên hàm sẽ tự dò.

In [ ]:
!pip install -q GEOparse

In [ ]:
import GEOparse

SYMBOL_CANDIDATES = ["Gene Symbol", "GENE_SYMBOL", "Symbol", "gene_symbol",
                     "ILMN_Gene", "GeneSymbol", "gene_assignment"]

def probe_to_symbol(gpl_id, destdir="/content/gpl"):
    os.makedirs(destdir, exist_ok=True)
    gpl = GEOparse.get_GEO(geo=gpl_id, destdir=destdir, silent=True)
    tbl = gpl.table
    col = next((c for c in SYMBOL_CANDIDATES if c in tbl.columns), None)
    if col is None:
        print(f"  {gpl_id}: khong tim thay cot symbol. Cac cot co san:")
        print("   ", list(tbl.columns))
        return None
    m = tbl[["ID", col]].dropna()
    m.columns = ["probe", "symbol"]
    m["symbol"] = m["symbol"].astype(str).str.split("///").str[0].str.strip()
    m = m[~m["symbol"].isin(["", "nan", "---"])]
    return m.set_index("probe")["symbol"]


def collapse_to_genes(expr, mapping):
    """Gop nhieu probe cua cung mot gene bang cach lay probe co phuong sai lon nhat."""
    sym = expr.index.map(mapping)
    keep = sym.notna()
    e = expr.loc[keep].copy()
    e["__sym"] = sym[keep]
    e["__var"] = expr.loc[keep].var(axis=1).values
    e = e.sort_values("__var", ascending=False).drop_duplicates("__sym")
    e = e.set_index("__sym").drop(columns="__var")
    e.index.name = "symbol"
    return e

In [ ]:
gene_level = {}

for gse, d in datasets.items():
    plats = sorted(set(d["pheno"]["Sample_platform_id"].dropna()))
    if len(plats) != 1:
        print(gse, "co nhieu platform, bo qua:", plats)
        continue
    gpl = plats[0]
    print(f"{gse} ({gpl}) ...", end=" ")
    mapping = probe_to_symbol(gpl)
    if mapping is None:
        continue
    g = collapse_to_genes(d["expr"], mapping)
    gene_level[gse] = g
    print(f"{g.shape[0]} gene x {g.shape[1]} mau")
    g.to_csv(os.path.join(OUTDIR, f"{gse}_gene.csv.gz"), compression="gzip")

## 9. Gộp các bộ (chỉ làm sau khi đã về gene symbol)

Gộp xong **bắt buộc phải hiệu chỉnh batch**, nếu không hiệu ứng batch sẽ lấn át
tín hiệu giai đoạn. Trong Python dùng `pycombat`; trong R dùng `sva::ComBat`.

In [ ]:
!pip install -q combat

In [ ]:
from functools import reduce

common_genes = reduce(lambda a, b: a & b,
                      [set(g.index) for g in gene_level.values()])
print("So gene chung:", len(common_genes))

common_genes = sorted(common_genes)
merged = pd.concat([g.loc[common_genes] for g in gene_level.values()], axis=1)
batch = pd.Series(
    sum([[gse] * g.shape[1] for gse, g in gene_level.items()], []),
    index=merged.columns)

print("Ma tran gop:", merged.shape)
print(batch.value_counts())

In [ ]:
from combat.pycombat import pycombat

merged_clean = merged.dropna()
corrected = pycombat(merged_clean, batch.loc[merged_clean.columns].values)
corrected.to_csv(os.path.join(OUTDIR, "merged_combat.csv.gz"), compression="gzip")

label_cols = ["stage", "stage_binary", "stage_encoded", "stage_binary_encoded"]
labels = pd.concat([d["pheno"][label_cols] for d in datasets.values()])
labels = labels.loc[[c for c in corrected.columns if c in labels.index]]
labels.to_csv(os.path.join(OUTDIR, "merged_labels.csv"))

print("Ma tran sau ComBat:", corrected.shape)
print(labels["stage_binary"].value_counts(dropna=False))

## 10. Ngắt kết nối Colab

`geo_expr.h5` và `geo_labels.csv` đã được lưu vào Google Drive. Cell dưới ngắt
runtime để giải phóng tài nguyên. Chỉ chạy cell này sau khi đã xác nhận 2 file
trên Drive đầy đủ.

In [ ]:
assert os.path.exists(H5_PATH) and os.path.exists(CSV_PATH), \
    "Chua thay file tren Drive, kiem tra lai truoc khi ngat ket noi."
print("Xac nhan da co tren Drive:", os.listdir(DRIVE_OUTDIR))

from google.colab import runtime
runtime.unassign()